In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/data-science-bowl-2018/stage1_test.zip
/kaggle/input/competitions/data-science-bowl-2018/stage1_sample_submission.csv.zip
/kaggle/input/competitions/data-science-bowl-2018/stage2_sample_submission_final.csv.zip
/kaggle/input/competitions/data-science-bowl-2018/stage1_train.zip
/kaggle/input/competitions/data-science-bowl-2018/stage1_train_labels.csv.zip
/kaggle/input/competitions/data-science-bowl-2018/stage1_solution.csv.zip
/kaggle/input/competitions/data-science-bowl-2018/stage2_test_final.zip


In [8]:
# ==============================================================================
# CELL 1: Extract Zip Files & Setup Working Directories
# ==============================================================================

import os
import zipfile

# 1. Define input zip paths
COMP_DIR = "/kaggle/input/competitions/data-science-bowl-2018"
TRAIN_ZIP = os.path.join(COMP_DIR, "stage1_train.zip")
TEST_ZIP = os.path.join(COMP_DIR, "stage1_test.zip")

# 2. Define output destination paths inside /kaggle/working/
EXTRACT_DIR = "/kaggle/working/dataset"
TRAIN_PATH = os.path.join(EXTRACT_DIR, "stage1_train")
TEST_PATH = os.path.join(EXTRACT_DIR, "stage1_test")

# 3. Create destination folders
os.makedirs(TRAIN_PATH, exist_ok=True)
os.makedirs(TEST_PATH, exist_ok=True)

# 4. Function to unzip files
def unzip_file(zip_src, extract_to):
    print(f"Extracting {os.path.basename(zip_src)}...")
    with zipfile.ZipFile(zip_src, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Done!")

# 5. Extract training and testing sets
unzip_file(TRAIN_ZIP, TRAIN_PATH)
unzip_file(TEST_ZIP, TEST_PATH)

# Verify extracted items count
train_ids = os.listdir(TRAIN_PATH)
test_ids = os.listdir(TEST_PATH)

print("\n--- EXTRACTION COMPLETE ---")
print(f"Extracted Training Samples: {len(train_ids)}")
print(f"Extracted Testing Samples:  {len(test_ids)}")

Extracting stage1_train.zip...
Done!
Extracting stage1_test.zip...
Done!

--- EXTRACTION COMPLETE ---
Extracted Training Samples: 670
Extracted Testing Samples:  65


In [9]:
# ==============================================================================
# CELL 2: Preprocess Images, Invert Colors & Merge Masks
# ==============================================================================

import glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# Set standardized image size
IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 3

WORKING_DIR = "/kaggle/working"

def process_sample(sample_id, base_dir, is_train=True):
    sample_dir = os.path.join(base_dir, sample_id)
    
    # --- Load Raw Image ---
    image_file = os.listdir(os.path.join(sample_dir, "images"))[0]
    img_path = os.path.join(sample_dir, "images", image_file)
    
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMG_WIDTH, IMG_HEIGHT))
    img_np = np.array(img, dtype=np.float32) / 255.0  # Normalize pixels [0, 1]
    
    # --- Invert Light Backgrounds ---
    # Makes all cells bright/white on a dark background
    if np.mean(img_np) > 0.5:
        img_np = 1.0 - img_np
        
    # --- Merge Split Masks (Train set only) ---
    if is_train:
        mask_files = glob.glob(os.path.join(sample_dir, "masks", "*.png"))
        combined_mask = np.zeros((IMG_HEIGHT, IMG_WIDTH, 1), dtype=np.float32)
        
        for m_path in mask_files:
            m = Image.open(m_path).convert("L").resize((IMG_WIDTH, IMG_HEIGHT))
            m_np = np.array(m, dtype=np.float32) / 255.0
            m_np = np.where(m_np > 0.5, 1.0, 0.0)  # Binary threshold
            
            # Merge current mask into total mask
            combined_mask[:, :, 0] = np.maximum(combined_mask[:, :, 0], m_np)
            
        return img_np, combined_mask
    else:
        return img_np


print("--- PREPROCESSING DATASET ---")

# Memory arrays for training set
X_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
Y_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, 1), dtype=np.float32)

for idx, sample_id in enumerate(tqdm(train_ids, desc="Processing Train Data")):
    img_proc, mask_proc = process_sample(sample_id, TRAIN_PATH, is_train=True)
    X_train[idx] = img_proc
    Y_train[idx] = mask_proc

# Save preprocessed arrays to disk for fast training
np.save(os.path.join(WORKING_DIR, "X_train.npy"), X_train)
np.save(os.path.join(WORKING_DIR, "Y_train.npy"), Y_train)

print(f"\nSuccessfully created and saved:")
print(f" -> X_train.npy shape: {X_train.shape}")
print(f" -> Y_train.npy shape: {Y_train.shape}")

--- PREPROCESSING DATASET ---


Processing Train Data:   0%|          | 0/670 [00:00<?, ?it/s]


Successfully created and saved:
 -> X_train.npy shape: (670, 256, 256, 3)
 -> Y_train.npy shape: (670, 256, 256, 1)


In [10]:
# ==============================================================================
# CELL 3: PyTorch Dataset Creation, Data Augmentations & DataLoaders
# ==============================================================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# Set random seed for reproducible train/val splits
torch.manual_seed(42)
np.random.seed(42)

# ------------------------------------------------------------------------------
# 1. Define Custom PyTorch Dataset Class
# ------------------------------------------------------------------------------
class NucleiDataset(Dataset):
    def __init__(self, x_data, y_data, is_train=True):
        """
        x_data: NumPy array of images  [N, 256, 256, 3]
        y_data: NumPy array of masks   [N, 256, 256, 1]
        is_train: Boolean to toggle data augmentations
        """
        self.x_data = x_data
        self.y_data = y_data
        self.is_train = is_train

    def __len__(self):
        # Total number of images in this dataset
        return len(self.x_data)

    def __getitem__(self, idx):
        # Fetch 1 image and 1 mask by index
        img = self.x_data[idx].copy()
        mask = self.y_data[idx].copy()

        # --- Data Augmentations (Only applied during training) ---
        if self.is_train:
            # Random Horizontal Flip (50% probability)
            if np.random.rand() > 0.5:
                img = np.fliplr(img)
                mask = np.fliplr(mask)

            # Random Vertical Flip (50% probability)
            if np.random.rand() > 0.5:
                img = np.flipud(img)
                mask = np.flipud(mask)

            # Random 90-degree Rotation
            rot_k = np.random.randint(0, 4)
            if rot_k > 0:
                img = np.rot90(img, k=rot_k, axes=(0, 1))
                mask = np.rot90(mask, k=rot_k, axes=(0, 1))

        # --- PyTorch Format Conversion ---
        # NumPy uses:   [Height, Width, Channels]  -> (256, 256, 3)
        # PyTorch uses: [Channels, Height, Width] -> (3, 256, 256)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).permute(2, 0, 1)

        return img_tensor, mask_tensor

# ------------------------------------------------------------------------------
# 2. Load Preprocessed Data Arrays from Disk
# ------------------------------------------------------------------------------
X_data = np.load("/kaggle/working/X_train.npy")
Y_data = np.load("/kaggle/working/Y_train.npy")

# Instantiate the complete dataset
full_dataset = NucleiDataset(X_data, Y_data, is_train=True)

# ------------------------------------------------------------------------------
# 3. Create 80/20 Train and Validation Split
# ------------------------------------------------------------------------------
total_samples = len(full_dataset)
train_size = int(0.80 * total_samples)  # 80% = 536 images
val_size = total_samples - train_size    # 20% = 134 images

# Randomly split indices into training and validation sets
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Disable augmentations on the validation set so evaluation stays static
val_dataset.dataset.is_train = False

# ------------------------------------------------------------------------------
# 4. Wrap with PyTorch DataLoaders (Batching and Parallel Loading)
# ------------------------------------------------------------------------------
BATCH_SIZE = 16  # Processes 16 images together in a single GPU pass

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,      # Shuffle images every epoch
    num_workers=2,     # Parallel CPU workers for faster loading
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,     # Do not shuffle validation data
    num_workers=2, 
    pin_memory=True
)

print("--- DATALOADERS SETUP COMPLETE ---")
print(f"Total Dataset Samples: {total_samples}")
print(f" -> Training Samples:   {len(train_dataset)} ({len(train_loader)} batches)")
print(f" -> Validation Samples: {len(val_dataset)} ({len(val_loader)} batches)")

# Inspect batch tensor dimensions
sample_imgs, sample_masks = next(iter(train_loader))
print(f"\nSample Batch Dimensions:")
print(f" -> Images batch shape (B, C, H, W): {sample_imgs.shape}")
print(f" -> Masks batch shape  (B, C, H, W): {sample_masks.shape}")

--- DATALOADERS SETUP COMPLETE ---
Total Dataset Samples: 670
 -> Training Samples:   536 (34 batches)
 -> Validation Samples: 134 (9 batches)

Sample Batch Dimensions:
 -> Images batch shape (B, C, H, W): torch.Size([16, 3, 256, 256])
 -> Masks batch shape  (B, C, H, W): torch.Size([16, 1, 256, 256])


In [11]:
# ==============================================================================
# CELL 4: Define U-Net Architecture & Dice-BCE Composite Loss
# ==============================================================================

import torch
import torch.nn as nn

# Detect hardware accelerator (GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ------------------------------------------------------------------------------
# 1. Building Block: Double Convolution Layer
# ------------------------------------------------------------------------------
class DoubleConv(nn.Module):
    """
    Applies two sequential [Conv2D -> BatchNorm -> ReLU] operations.
    Keeps spatial dimensions equal via padding=1.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


# ------------------------------------------------------------------------------
# 2. Main Model: U-Net Architecture
# ------------------------------------------------------------------------------
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        
        # --- ENCODER (Contracting Path) ---
        self.enc1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 256x256 -> 128x128
        
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 128x128 -> 64x64
        
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 64x64 -> 32x32

        # --- BOTTLENECK ---
        self.bottleneck = DoubleConv(256, 512)              # Deepest features (32x32)

        # --- DECODER (Expanding Path) ---
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2) # 32x32 -> 64x64
        self.dec3 = DoubleConv(512, 256)                    # 256 (from up) + 256 (skip) = 512
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2) # 64x64 -> 128x128
        self.dec2 = DoubleConv(256, 128)                    # 128 (from up) + 128 (skip) = 256
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)  # 128x128 -> 256x256
        self.dec1 = DoubleConv(128, 64)                     # 64 (from up) + 64 (skip) = 128

        # --- FINAL OUTPUT CONVOLUTION ---
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder passes (Save skip features before max-pooling)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        # Bottleneck
        b = self.bottleneck(self.pool3(e3))

        # Decoder passes with Skip Connections (Concatenation along Channel axis dim=1)
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Final prediction layer (Outputs raw logits for numerically stable Sigmoid Loss)
        return self.final_conv(d1)


# ------------------------------------------------------------------------------
# 3. Loss Function: Dice + BCE Loss Combo
# ------------------------------------------------------------------------------
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        # BCEWithLogitsLoss combines Sigmoid + BCE for high numerical stability
        self.bce_logits = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets, smooth=1e-6):
        # 1. Compute Binary Cross-Entropy Loss
        bce_loss = self.bce_logits(logits, targets)

        # 2. Convert raw logits to probabilities via Sigmoid for Dice calculation
        probs = torch.sigmoid(logits)
        
        # Flatten tensors to 1D vectors
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)

        # 3. Compute Dice Coefficient Loss
        intersection = (probs_flat * targets_flat).sum()
        dice_score = (2. * intersection + smooth) / (probs_flat.sum() + targets_flat.sum() + smooth)
        dice_loss = 1.0 - dice_score

        # Combine both losses equally
        return bce_loss + dice_loss


# Instantiate network and move parameters to GPU memory
model = UNet(in_channels=3, out_channels=1).to(device)
criterion = DiceBCELoss()

print("--- U-NET ARCHITECTURE READY ---")
print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Verification check: Pass dummy tensor to verify shape matches input/output
dummy_input = torch.randn(2, 3, 256, 256).to(device)
dummy_output = model(dummy_input)
print(f"Dummy Pass Shape Test: Input {tuple(dummy_input.shape)} -> Output {tuple(dummy_output.shape)}")

Using device: cuda
--- U-NET ARCHITECTURE READY ---
Total Model Parameters: 7,702,977
Dummy Pass Shape Test: Input (2, 3, 256, 256) -> Output (2, 1, 256, 256)


In [12]:
# ==============================================================================
# CELL 5: Training Loop, Validation Metric (IoU) & Model Checkpointing
# ==============================================================================

import time
import torch
import torch.optim as optim

# ------------------------------------------------------------------------------
# 1. Metric Definition: Mean Intersection over Union (IoU)
# ------------------------------------------------------------------------------
def compute_iou(logits, targets, threshold=0.5, smooth=1e-6):
    """
    Computes IoU (Jaccard Index) for binary predictions.
    IoU = Overlap Area / Union Area
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    
    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)
    
    intersection = (preds_flat * targets_flat).sum()
    total = preds_flat.sum() + targets_flat.sum()
    union = total - intersection
    
    iou = (intersection + smooth) / (union + smooth)
    return iou.item()

# ------------------------------------------------------------------------------
# 2. Hyperparameters & Optimizer Setup
# ------------------------------------------------------------------------------
EPOCHS = 20
LEARNING_RATE = 1e-4

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Track best validation IoU for model checkpointing
best_val_iou = 0.0
checkpoint_path = "/kaggle/working/unet_best.pth"

print(f"--- STARTING TRAINING FOR {EPOCHS} EPOCHS ---")
start_time = time.time()

# ------------------------------------------------------------------------------
# 3. Main Training Loop
# ------------------------------------------------------------------------------
for epoch in range(1, EPOCHS + 1):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    train_iou = 0.0
    
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()            # Clear previous gradients
        outputs = model(images)          # Forward pass
        loss = criterion(outputs, masks) # Compute loss
        loss.backward()                  # Backward pass
        optimizer.step()                 # Update weights
        
        train_loss += loss.item()
        train_iou += compute_iou(outputs, masks)
        
    avg_train_loss = train_loss / len(train_loader)
    avg_train_iou = train_iou / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    
    with torch.no_grad(): # Disable gradient calculation for faster evaluation
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            val_loss += loss.item()
            val_iou += compute_iou(outputs, masks)
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou = val_iou / len(val_loader)
    
    # --- PRINT EPOCH SUMMARY ---
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
          f"| Train Loss: {avg_train_loss:.4f} | Train IoU: {avg_train_iou:.4f} "
          f"| Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f}")
    
    # --- SAVE BEST MODEL CHECKPOINT ---
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  --> Checkpoint Saved! New Best Val IoU: {best_val_iou:.4f}")

total_elapsed = time.time() - start_time
print(f"\nTraining Complete in {total_elapsed/60:.2f} minutes!")
print(f"Best Saved Model Weights: {checkpoint_path} (Val IoU: {best_val_iou:.4f})")

--- STARTING TRAINING FOR 20 EPOCHS ---
Epoch [01/20] | Train Loss: 1.1142 | Train IoU: 0.4649 | Val Loss: 1.2744 | Val IoU: 0.3967
  --> Checkpoint Saved! New Best Val IoU: 0.3967
Epoch [02/20] | Train Loss: 0.8792 | Train IoU: 0.6915 | Val Loss: 0.7912 | Val IoU: 0.7253
  --> Checkpoint Saved! New Best Val IoU: 0.7253
Epoch [03/20] | Train Loss: 0.8103 | Train IoU: 0.7447 | Val Loss: 0.7514 | Val IoU: 0.7629
  --> Checkpoint Saved! New Best Val IoU: 0.7629
Epoch [04/20] | Train Loss: 0.7582 | Train IoU: 0.7763 | Val Loss: 0.7126 | Val IoU: 0.7916
  --> Checkpoint Saved! New Best Val IoU: 0.7916
Epoch [05/20] | Train Loss: 0.7241 | Train IoU: 0.7860 | Val Loss: 0.6909 | Val IoU: 0.7653
Epoch [06/20] | Train Loss: 0.7012 | Train IoU: 0.7808 | Val Loss: 0.6879 | Val IoU: 0.7637
Epoch [07/20] | Train Loss: 0.6709 | Train IoU: 0.7922 | Val Loss: 0.6365 | Val IoU: 0.7890
Epoch [08/20] | Train Loss: 0.6412 | Train IoU: 0.7969 | Val Loss: 0.6157 | Val IoU: 0.8198
  --> Checkpoint Saved! New 

In [14]:
# ==============================================================================
# CELL 6: Test Inference, Watershed Instance Segmentation & RLE CSV Export
# ==============================================================================

import os
import glob
import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image
from tqdm.notebook import tqdm

# ------------------------------------------------------------------------------
# 1. Load Trained Weights & Set Model to Evaluation Mode
# ------------------------------------------------------------------------------
model.load_state_dict(torch.load("/kaggle/working/unet_best.pth"))
model.eval()

# ------------------------------------------------------------------------------
# 2. Run-Length Encoding (RLE) Function
# ------------------------------------------------------------------------------
def rle_encode(mask):
    """
    Encodes binary 2D mask array into Run-Length Encoding string.
    Note: Kaggle expects column-first order (Fortran 'F' order).
    """
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return ' '.join(str(x) for x in runs)

# ------------------------------------------------------------------------------
# 3. Watershed Separation for Touching Nuclei
# ------------------------------------------------------------------------------
def separate_touching_nuclei(binary_mask):
    """
    Applies distance transform + Watershed algorithm to separate touching nuclei.
    Returns a list of individual binary masks (one per nucleus instance).
    """
    mask_uint8 = (binary_mask * 255).astype(np.uint8)
    
    # Distance transform calculates distance from each white pixel to nearest black pixel
    dist_transform = cv2.distanceTransform(mask_uint8, cv2.DIST_L2, 5)
    
    # Threshold distance map to extract core nucleus centers (sure foreground)
    _, sure_fg = cv2.threshold(dist_transform, 0.3 * dist_transform.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    
    # Connected components on sure foreground markers
    _, markers = cv2.connectedComponents(sure_fg)
    
    # Apply Watershed on top of distance map
    dist_3ch = cv2.cvtColor(mask_uint8, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(dist_3ch, markers)
    
    # Extract unique instance labels (excluding background label 0/1 and boundaries -1)
    unique_labels = np.unique(markers)
    individual_masks = []
    
    for label in unique_labels:
        if label <= 0:  # Skip background and watershed borders
            continue
        instance_mask = (markers == label).astype(np.uint8)
        if instance_mask.sum() > 10:  # Filter out tiny noisy artifacts
            individual_masks.append(instance_mask)
            
    return individual_masks

# ------------------------------------------------------------------------------
# 4. Predict Test Data & Create Submission Entries
# ------------------------------------------------------------------------------
rle_records = []

print("--- RUNNING TEST INFERENCE & RLE ENCODING ---")

for sample_id in tqdm(test_ids, desc="Processing Test Samples"):
    sample_dir = os.path.join(TEST_PATH, sample_id)
    image_file = os.listdir(os.path.join(sample_dir, "images"))[0]
    img_path = os.path.join(sample_dir, "images", image_file)
    
    # Load original image and store native shape for rescaling back later
    raw_img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = raw_img.size
    
    # Preprocess & Resize to 256x256
    resized_img = raw_img.resize((IMG_WIDTH, IMG_HEIGHT))
    img_np = np.array(resized_img, dtype=np.float32) / 255.0
    
    if np.mean(img_np) > 0.5:
        img_np = 1.0 - img_np
        
    # Tensor Conversion & Model Prediction
    input_tensor = torch.tensor(img_np, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
        pred_prob = torch.sigmoid(logits).squeeze().cpu().numpy()
        
    # Threshold probability map to binary prediction
    binary_pred = (pred_prob > 0.5).astype(np.uint8)
    
    # Resize binary prediction back to original image dimensions
    binary_orig = cv2.resize(binary_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    
    # Separate touching nuclei instances using Watershed
    instance_masks = separate_touching_nuclei(binary_orig)
    
    # If Watershed found instances, encode each separately; else encode full binary mask
    if len(instance_masks) > 0:
        for inst_m in instance_masks:
            rle_str = rle_encode(inst_m)
            if len(rle_str) > 0:
                rle_records.append({'ImageId': sample_id, 'EncodedPixels': rle_str})
    else:
        rle_str = rle_encode(binary_orig)
        rle_records.append({'ImageId': sample_id, 'EncodedPixels': rle_str})

# ------------------------------------------------------------------------------
# 5. Save Final Submission CSV File
# ------------------------------------------------------------------------------
submission_df = pd.DataFrame(rle_records)
submission_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_path, index=False)

print("\n--- SUBMISSION FILE GENERATED SUCCESSFULLY ---")
print(f"Saved: {submission_path}")
print(f"Total Prediction Rows: {len(submission_df)}")
print("\nFirst 5 Rows Preview:")
print(submission_df.head())

--- RUNNING TEST INFERENCE & RLE ENCODING ---


Processing Test Samples:   0%|          | 0/65 [00:00<?, ?it/s]


--- SUBMISSION FILE GENERATED SUCCESSFULLY ---
Saved: /kaggle/working/submission.csv
Total Prediction Rows: 2701

First 5 Rows Preview:
                                             ImageId  \
0  bdc789019cee8ddfae20d5f769299993b4b330b2d38d12...   
1  bdc789019cee8ddfae20d5f769299993b4b330b2d38d12...   
2  bdc789019cee8ddfae20d5f769299993b4b330b2d38d12...   
3  bdc789019cee8ddfae20d5f769299993b4b330b2d38d12...   
4  bdc789019cee8ddfae20d5f769299993b4b330b2d38d12...   

                                       EncodedPixels  
0  522 518 1042 518 1562 518 2082 518 2602 518 31...  
1  115962 8 116482 8 117002 9 117522 12 118042 12...  
2  288606 8 289126 8 289645 10 290164 14 290684 1...  
3  180986 14 181506 14 182025 16 182542 24 183062...  
4  92096 17 92616 17 93135 19 93645 30 94164 31 9...  
